# R-Learner Uplift Model

This notebook implements the R-learner (`causalml.inference.meta.BaseRClassifier`) for the same binary any-email treatment setup used throughout this project: `treatment = 1` for customers who received either the Mens or Womens e-mail, `treatment = 0` for customers who received no e-mail.

**Why R-learner, and why now.** The X-learner notebook (`06_x_learner.ipynb`) diagnosed a specific, well-documented weakness: its stage-2 regression target is an individual *imputed* pseudo-effect (`y - m̂(x)`), which inherits the full Bernoulli noise of a binary outcome (`Var(y) = p(1-p)`, empirically ~0.31-0.37 SD here) even though the true effects being estimated are much smaller (0.02-0.10). We proved this empirically -- across 3 model families, both regularization directions, an explicit engineered interaction feature, and an aggressive sample-weight sweep up to 500x -- and it persisted even with *zero* ensembling (a plain linear model showed essentially the identical underestimate of a real, statistically verified segment effect as Random Forest). The literature (Nie & Wager 2017) proposes the R-learner specifically to address this: it residualizes both the outcome and the treatment (the Robinson decomposition) and fits the effect model on a single, lower-variance target via a weighted "R-loss," rather than X-learner's two-sided imputation. This notebook tests whether that actually holds up on this dataset, with a direct, apples-to-apples comparison against the X-learner's already-established results.

This notebook carries forward validated findings from the X-learner notebook rather than rediscovering them: the `mens_and_womens` engineered interaction feature, the same three regularized model families (linear / random_forest / lightgbm) compared via 5-fold out-of-fold CV to avoid test-set leakage, the same decile / segment / Qini / AUUC / ROI evaluation pipeline (via `src/metrics.py`, `src/plotting.py`, `src/roi.py`), the same empirically-grounded ROI assumptions (`profit_per_conversion` from train-split `spend`, `response_baseline`'s `logistic_proba` for the response-probability targeting strategy), and the pooled train+test benchmark for the `mens_and_womens` segment effect (~10.3pp) after X-learner's notebook found the original test-only estimate (16.0pp) was inflated by a selection effect.

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path

# BaseRClassifier's internal cross_val_predict(..., n_jobs=-1) spawns joblib worker
# subprocesses for the outcome model that do NOT inherit this process's in-memory
# warnings.filterwarnings() state -- only environment variables carry over. Setting
# PYTHONWARNINGS here (before any subprocess gets spawned) is what actually reaches
# those workers; the filterwarnings() calls below still handle everything in this
# main process.
os.environ["PYTHONWARNINGS"] = "ignore"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression, ElasticNetCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier, LGBMRegressor
from joblib import parallel_backend

from causalml.inference.meta import BaseRClassifier
from causalml.propensity import ElasticNetPropensityModel

pd.set_option("display.max_columns", 100)

# Same warning sources as 06_x_learner.ipynb: unscaled-feature solver instability
# (verified benign there; CV/calibration correct for it) and LightGBM's placeholder
# feature-name mismatch on numpy-array predict() calls (cosmetic only).
warnings.filterwarnings("ignore", category=RuntimeWarning, module=r"sklearn\..*")
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names", category=UserWarning)

In [ ]:
# Allows notebook to import from src/ when running inside notebooks/
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    PROCESSED_DATA_DIR,
    PREDICTIONS_DIR,
    MODEL_RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    RANDOM_STATE,
)
from src.metrics import (
    build_decile_table,
    build_segment_table,
    compute_qini_auuc,
    propensity_matches_known,
    segments_agree_across_samples,
    uplift_at_top_k,
    stratified_uplift_folds,
)
from src.plotting import (
    save_fig,
    plot_qini_curve,
    plot_cumulative_gain_curve,
    plot_decile_lift_bar,
    plot_propensity_diagnostic,
)
from src.roi import compare_targeting_strategies

for directory in (PREDICTIONS_DIR, MODEL_RESULTS_DIR, FIGURES_DIR, TABLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Processed data dir:", PROCESSED_DATA_DIR)
print("Predictions dir:", PREDICTIONS_DIR)
print("Model results dir:", MODEL_RESULTS_DIR)
print("Figures dir:", FIGURES_DIR)
print("Tables dir:", TABLES_DIR)

### Load processed train/test data

Same processed files as every other model notebook in this project, created by `02_feature_matrix.ipynb`. `treatment_binary` is loaded positionally, not by name (see `06_x_learner.ipynb` for why: it's a deliberate, committed mismatch with `src.config.TREATMENT_COL`).

In [ ]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")

y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").iloc[:, 0]
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").iloc[:, 0]

treatment_train = pd.read_csv(PROCESSED_DATA_DIR / "treatment_train.csv").iloc[:, 0]
treatment_test = pd.read_csv(PROCESSED_DATA_DIR / "treatment_test.csv").iloc[:, 0]

secondary_y_train = pd.read_csv(PROCESSED_DATA_DIR / "secondary_y_train.csv")
secondary_y_test = pd.read_csv(PROCESSED_DATA_DIR / "secondary_y_test.csv")

segment_test = pd.read_csv(PROCESSED_DATA_DIR / "segment_test.csv").iloc[:, 0]

assert len(X_train) == len(y_train) == len(treatment_train) == len(secondary_y_train)
assert len(X_test) == len(y_test) == len(treatment_test) == len(secondary_y_test) == len(segment_test)
assert set(treatment_train.unique()) <= {0, 1}
assert set(treatment_test.unique()) <= {0, 1}

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Treatment column name on disk:", treatment_train.name)

### Known randomized propensity

Per this project's propensity strategy (README): R-learner *requires* a propensity score as part of its loss construction (`weight = (T - ê(X))²`), so this isn't optional here the way it partly was for X-learner. Because assignment is randomized (two of three original arms collapsed into `treatment=1`), the known design propensity is `p = P(T=1) ≈ 2/3`, computed directly from data rather than hardcoded.

In [ ]:
p_known = float(treatment_train.mean())

print(f"Known randomized propensity (train treatment rate): {p_known:.4f}")
assert abs(p_known - 2 / 3) < 0.05, "Observed treatment rate is not close to the expected ~2/3." 

### Robustness check: CausalML's estimated propensity

Same robustness check as `06_x_learner.ipynb`: fit `ElasticNetPropensityModel` and confirm it recovers the known ~2/3 rate with no strong feature-driven structure, as expected under randomization. The known propensity remains the primary input to R-learner's `.fit()` -- this is a comparison/validation check, not the source of identification.

In [ ]:
propensity_model = ElasticNetPropensityModel(calibrate=True, random_state=RANDOM_STATE, n_fold=3)
propensity_model.fit(X_train, treatment_train)

p_hat_train = propensity_model.predict(X_train)
p_hat_test = propensity_model.predict(X_test)

print("Estimated propensity (test) summary:")
print(pd.Series(p_hat_test).describe())

In [ ]:
propensity_check = propensity_matches_known(p_hat_test, p_known, tolerance=0.01)

print(f"Known propensity:     {propensity_check['known_p']:.4f}")
print(f"Estimated mean p:     {propensity_check['mean_p']:.4f}")
print(f"Estimated std p:      {propensity_check['std_p']:.4f}")
print(f"|diff|:               {propensity_check['diff']:.4f}  (tolerance {propensity_check['tolerance']:.3f})")
print(f"Agreement verdict:    {'PASS' if propensity_check['agrees'] else 'FAIL'}")

assert propensity_check["agrees"], (
    "Estimated propensity does not agree with the known randomized propensity -- "
    "investigate before trusting the known-propensity R-loss weighting below."
)

In [ ]:
fig_propensity = plot_propensity_diagnostic(p_hat_test, known_p=p_known)
save_fig(fig_propensity, "r_learner_propensity_diagnostic.png")

### Engineered feature: mens_and_womens interaction

Carried forward from `06_x_learner.ipynb`: a segment-level diagnostic there found that customers who historically purchased *both* mens and womens merchandise (~10% of customers) show a real uplift meaningfully above the other segments. X-learner underestimated this segment regardless of model family, regularization, or sample weighting. We add the same `mens_and_womens = mens * womens` feature here so R-learner gets the same direct access to it -- this keeps the comparison between the two algorithms about the *algorithm*, not about which one happened to get a better feature set. In memory only; `data/processed/` on disk is untouched.

**Note**: `06_x_learner.ipynb` originally benchmarked this segment against its test-set-only estimate (16.0pp), but a later check found that estimate didn't replicate on train (8.4pp there, z=3.12 disagreement) -- a selection effect from having found the segment by searching test-set correlations in the first place. The corrected, pooled (train+test) benchmark is ~10.3pp; this notebook uses that corrected figure throughout.

In [ ]:
X_train["mens_and_womens"] = X_train["mens"] * X_train["womens"]
X_test["mens_and_womens"] = X_test["mens"] * X_test["womens"]

print("Added mens_and_womens interaction feature.")
print(f"  Train: {int(X_train['mens_and_womens'].sum())} / {len(X_train)} customers bought both")
print(f"  Test:  {int(X_test['mens_and_womens'].sum())} / {len(X_test)} customers bought both")

### How R-learner differs from X-learner mechanically

`causalml.inference.meta.BaseRClassifier` implements the Nie & Wager (2017) R-learner. Two structural differences from `BaseXClassifier` matter for this comparison:

1. **A single, pooled outcome model with built-in cross-fitting.** X-learner fits *separate* `m̂₀`/`m̂₁` on the control-only and treated-only subsets, using the same data for fitting and for imputing pseudo-effects. R-learner fits *one* `m̂(x) = E[Y|X]` pooling both arms, via `sklearn.model_selection.cross_val_predict` -- every prediction used downstream comes from a fold where that row was held out during fitting. This is "honest" estimation for the outcome nuisance model, built into the algorithm rather than bolted on.
2. **A single unified effect model per arm, fit via the R-loss**, not two cross-imputed models combined by propensity weighting at predict time. The regression target is `(y - m̂(x)) / (T - ê(x))`, weighted by `(T - ê(x))²` -- algebraically equivalent to minimizing `Σ(Ỹᵢ - τ(Xᵢ)ᵀ̃ᵢ)²` (the Robinson/R-loss). Since propensity is known and constant here (≈0.667), this weighting is a fixed, non-degenerate `(1/3)²` vs. `(2/3)²` split between control and treated rows -- not estimated.

Because the R-loss requires `effect_learner.fit(X, target, sample_weight=weight)` internally (not optional), every `effect_learner` here must support `sample_weight` directly. `LogisticRegression`/`RandomForestRegressor`/`LGBMRegressor` all do natively; a plain `sklearn.pipeline.Pipeline` does *not* accept an unprefixed `sample_weight` kwarg the way `BaseRClassifier` calls it, so the scaled-linear family below uses a small custom wrapper instead of `Pipeline`.

### Compare model families for outcome_learner / effect_learner

Same three matched families as `06_x_learner.ipynb`, with the same regularization choices already validated there (loosening them was tested and made things worse, not better -- see that notebook's "Compare model families" section):

- **linear**: `LogisticRegression` (outcome) + `ElasticNetCV` (effect), features standardized. Unlike X-learner's original linear family, this is the *corrected* version from the start (scaled features, CV-selected `alpha`) -- X-learner's notebook found that skipping scaling silently breaks ElasticNet's regularization budget entirely, so there's no reason to repeat that mistake here.
- **random_forest**: `RandomForestClassifier` + `RandomForestRegressor`, `max_depth=6`, `min_samples_leaf=50`.
- **lightgbm**: `LGBMClassifier` + `LGBMRegressor`, `max_depth=4`, `num_leaves=15`, `min_child_samples=100`, `reg_alpha`/`reg_lambda=0.1`.

**Selection methodology**, unchanged from X-learner: picking the winner on `X_test` would double-dip the test set. 5-fold cross-validation within `X_train` gives every training row exactly one out-of-fold prediction per family; one Qini score per family is computed from the full stitched-together out-of-fold predictions (48,000 rows), and the winner is refit on the full training set and evaluated once on the untouched `X_test`.

In [ ]:
N_FOLDS = 5
p_known_train = np.full(len(X_train), p_known)

print(f"{N_FOLDS}-fold CV within X_train ({len(X_train)} rows), stratified jointly by treatment and outcome.")

In [ ]:
class ScaledEffectRegressor:
    """Wraps a regressor with feature standardization while still accepting a raw
    `sample_weight` kwarg in .fit() -- a plain sklearn Pipeline requires the prefixed
    `stepname__sample_weight` form, but BaseRClassifier's R-loss fitting calls
    effect_learner.fit(X, target, sample_weight=weight) directly, unprefixed.
    """

    def __init__(self, model):
        self.model = model
        self.scaler = StandardScaler()

    def fit(self, X, y, sample_weight=None):
        Xs = self.scaler.fit_transform(X)
        if sample_weight is not None:
            self.model.fit(Xs, y, sample_weight=sample_weight)
        else:
            self.model.fit(Xs, y)
        return self

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))


def make_linear_family():
    return (
        Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
        ScaledEffectRegressor(ElasticNetCV(l1_ratio=0.5, cv=5, n_alphas=50, random_state=RANDOM_STATE)),
    )


def make_random_forest_family():
    return (
        RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=50, random_state=RANDOM_STATE),
        RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=50, random_state=RANDOM_STATE),
    )


def make_lightgbm_family():
    lgbm_kwargs = dict(
        random_state=RANDOM_STATE, verbose=-1, n_estimators=200, max_depth=4,
        num_leaves=15, min_child_samples=100, learning_rate=0.05, reg_alpha=0.1, reg_lambda=0.1,
    )
    return (LGBMClassifier(**lgbm_kwargs), LGBMRegressor(**lgbm_kwargs))


model_families = {
    "linear": make_linear_family,
    "random_forest": make_random_forest_family,
    "lightgbm": make_lightgbm_family,
}

tau_hat_oof_by_family = {name: np.full(len(X_train), np.nan) for name in model_families}

for family_name, make_family in model_families.items():
    for fold_train_idx, fold_val_idx in stratified_uplift_folds(
    y_train, treatment_train, n_splits=N_FOLDS, random_state=RANDOM_STATE
):
        X_fold_train, X_fold_val = X_train.iloc[fold_train_idx], X_train.iloc[fold_val_idx]
        y_fold_train = y_train.iloc[fold_train_idx]
        treatment_fold_train = treatment_train.iloc[fold_train_idx]
        p_fold_train = p_known_train[fold_train_idx]

        outcome_learner, effect_learner = make_family()
        fold_r_learner = BaseRClassifier(outcome_learner=outcome_learner, effect_learner=effect_learner, control_name=0, n_fold=3, random_state=RANDOM_STATE)
        # CausalML 0.15.5 hard-codes n_jobs=-1 for this internal
        # cross_val_predict. Use threads so constrained notebook runtimes do
        # not need to spawn worker processes; model results are unchanged.
        with parallel_backend("threading"):
            fold_r_learner.fit(X_fold_train, treatment_fold_train, y_fold_train, p=p_fold_train, verbose=False)
        tau_hat_oof_by_family[family_name][fold_val_idx] = fold_r_learner.predict(X_fold_val).ravel()

family_oof_scores = compute_qini_auuc(y_train, treatment_train, tau_hat_oof_by_family)
family_comparison = pd.DataFrame({
    "qini_score": family_oof_scores["qini_score"],
    "auuc_score": family_oof_scores["auuc_score"],
}).sort_values("qini_score", ascending=False)

print(f"Model family comparison ({N_FOLDS}-fold out-of-fold predictions, full X_train):")
display(family_comparison)

winning_family = family_comparison.index[0]
print(f"\nWinning family (highest out-of-fold Qini): {winning_family}")

### Refit the winning family on the full training set

The winning family is refit on all of `X_train` and evaluated once on the untouched `X_test`, producing the canonical `tau_hat_r` that the rest of this notebook saves and plots. `BaseRClassifier.predict()` accepts a `p` argument for API symmetry with other learners but ignores it -- propensity is already baked into the fitted `models_tau` via the R-loss weighting, not reapplied at predict time (unlike X-learner's `p*dhat_c + (1-p)*dhat_t` combination).

In [ ]:
outcome_learner, effect_learner = model_families[winning_family]()
r_learner = BaseRClassifier(outcome_learner=outcome_learner, effect_learner=effect_learner, control_name=0, n_fold=5, random_state=RANDOM_STATE)
with parallel_backend("threading"):
    r_learner.fit(X_train, treatment_train, y_train, p=p_known_train, verbose=False)
tau_hat_r = r_learner.predict(X_test).ravel()

print(f"tau_hat_r summary (winning family: {winning_family}):")
print(pd.Series(tau_hat_r).describe())

### Save `tau_hat_r` predictions

Saved to `outputs/predictions/tau_hat_r.csv` for downstream evaluation and cross-learner comparison notebooks.

In [ ]:
tau_hat_r_df = pd.DataFrame({
    "test_row_id": np.arange(len(X_test)),
    "y_true": y_test.to_numpy(),
    "treatment": treatment_test.to_numpy(),
    "tau_hat_r": tau_hat_r,
})
tau_hat_r_oof_df = pd.DataFrame({
    "train_row_id": np.arange(len(X_train)),
    "y_true": y_train.to_numpy(),
    "treatment": treatment_train.to_numpy(),
    "tau_hat_r": tau_hat_oof_by_family[winning_family],
})

test_path = PREDICTIONS_DIR / "tau_hat_r.csv"
oof_path = PREDICTIONS_DIR / "tau_hat_r_oof.csv"
tau_hat_r_df.to_csv(test_path, index=False)
tau_hat_r_oof_df.to_csv(oof_path, index=False)

print("Saved test predictions:", test_path)
print("Saved OOF predictions:", oof_path)


### Uplift-by-decile evaluation

Same methodology as `06_x_learner.ipynb`: rank by descending `tau_hat_r`, split into 10 equal-count bins, report each decile's realized lift with a 95% CI (binomial-style standard error via `src.metrics._slice_lift`).

In [ ]:
decile_table = build_decile_table(y_test, treatment_test, tau_hat_r, n_bins=10)
display(decile_table)

decile_table.to_csv(TABLES_DIR / "r_learner_decile_table.csv")
print("Saved:", TABLES_DIR / "r_learner_decile_table.csv")

In [ ]:
deciles_only = decile_table.drop(index="Overall")

print("Adjacent-decile 95% CI overlap (overlap => can't statistically distinguish the two deciles' lift):")
n_overlapping = 0
for b in range(1, 10):
    a, c = deciles_only.loc[b], deciles_only.loc[b + 1]
    overlaps = (a["actual_lift_ci_lower"] <= c["actual_lift_ci_upper"]) and (c["actual_lift_ci_lower"] <= a["actual_lift_ci_upper"])
    n_overlapping += overlaps
    print(f"  Decile {b} vs {b + 1}: {'OVERLAP' if overlaps else 'distinguishable'}")

print(f"\n{n_overlapping} / 9 adjacent-decile pairs have overlapping CIs.")

top, bottom = deciles_only.loc[1], deciles_only.loc[10]
top_vs_bottom_overlap = (top["actual_lift_ci_lower"] <= bottom["actual_lift_ci_upper"]) and (bottom["actual_lift_ci_lower"] <= top["actual_lift_ci_upper"])
print(f"Decile 1 vs decile 10 (top vs. bottom): {'OVERLAP' if top_vs_bottom_overlap else 'distinguishable'}")

### Segment diagnostic: mens/womens purchase history

This is the key test. `06_x_learner.ipynb` found the `both` segment's pooled (train+test) true lift is ~10.3pp, underestimated by every X-learner variant tried (~7.9pp predicted, including with this exact interaction feature given directly). If R-learner's lower-variance stage-2 target actually helps, `mean_tau_hat` for the `both` segment here should sit closer to 10.3pp than X-learner's 7.9pp did. The cells below also independently recompute the train/pooled benchmark from this notebook's own data, rather than trusting a retyped number.

In [ ]:
segment_labels = pd.Series(np.select(
    [
        (X_test["mens"] == 1) & (X_test["womens"] == 1),
        (X_test["mens"] == 1) & (X_test["womens"] == 0),
        (X_test["mens"] == 0) & (X_test["womens"] == 1),
    ],
    ["both", "mens_only", "womens_only"],
    default="neither",
), index=X_test.index)
assert (segment_labels == "neither").sum() == 0, "Found customers who bought neither mens nor womens merchandise."

segment_table = build_segment_table(
    y_test, treatment_test, segment_labels,
    segment_order=["womens_only", "mens_only", "both"],
    tau_hat=tau_hat_r,
)
display(segment_table)

segment_table.to_csv(TABLES_DIR / "r_learner_segment_table.csv")
print("Saved:", TABLES_DIR / "r_learner_segment_table.csv")

fig_segment_bar = plot_decile_lift_bar(
    segment_table,
    title="Actual lift by mens/womens purchase-history segment (R-learner)",
    xlabel="Segment (customer purchase history)",
)
save_fig(fig_segment_bar, "r_learner_segment_lift_bar.png")

### Does the `both` segment replicate on train? Pooled benchmark

Same consistency check as `06_x_learner.ipynb`: recompute the segment breakdown on `X_train` (never used to find this segment) and on `X_train`+`X_test` pooled, then run a two-sample z-test comparing the train and test `both`-segment estimates. Independently verifies the corrected ~10.3pp benchmark rather than trusting the number quoted above.

In [ ]:
segment_labels_train = pd.Series(np.select(
    [
        (X_train["mens"] == 1) & (X_train["womens"] == 1),
        (X_train["mens"] == 1) & (X_train["womens"] == 0),
        (X_train["mens"] == 0) & (X_train["womens"] == 1),
    ],
    ["both", "mens_only", "womens_only"],
    default="neither",
), index=X_train.index)

segment_table_train = build_segment_table(
    y_train, treatment_train, segment_labels_train,
    segment_order=["womens_only", "mens_only", "both"],
)
print("Segment table, TRAIN only (not used to find this segment):")
display(segment_table_train[["n_customers", "actual_lift", "actual_lift_ci_lower", "actual_lift_ci_upper"]])

both_consistency = segments_agree_across_samples(segment_table_train, segment_table, "both")
print(f"\nTrain 'both' estimate:  {both_consistency['estimate_a']:.4f}")
print(f"Test 'both' estimate:   {both_consistency['estimate_b']:.4f}")
print(f"z = {both_consistency['z']:.2f}  ({'agree' if both_consistency['agrees'] else 'DISAGREE -- test estimate likely inflated by selection effect'})")

X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)
treatment_all = pd.concat([treatment_train, treatment_test], ignore_index=True)
segment_labels_all = pd.concat([segment_labels_train, segment_labels], ignore_index=True)

segment_table_pooled = build_segment_table(
    y_all, treatment_all, segment_labels_all,
    segment_order=["womens_only", "mens_only", "both"],
)
print("\nSegment table, POOLED (train+test, n=64,000):")
display(segment_table_pooled[["n_customers", "actual_lift", "actual_lift_se", "actual_lift_ci_lower", "actual_lift_ci_upper"]])

segment_table_pooled.to_csv(TABLES_DIR / "r_learner_segment_table_pooled.csv")
print("Saved:", TABLES_DIR / "r_learner_segment_table_pooled.csv")

both_pooled_lift = float(segment_table_pooled.loc["both", "actual_lift"])
print(f"\nPooled 'both' segment lift (the benchmark used below): {both_pooled_lift:.4f}")

### Head-to-head: R-learner vs. X-learner on the segment underestimate

Loads `06_x_learner.ipynb`'s saved results directly (not retyped numbers) for an apples-to-apples comparison on the untouched test set. Uses this notebook's own pooled (train+test) segment table, computed above, as `true_actual_lift` -- since that's derived from the raw `y`/`treatment` data (not from either model), it doesn't depend on which notebook computed it, and pooling avoids the selection-effect inflation the test-only estimate had.

In [ ]:
x_learner_results_path = MODEL_RESULTS_DIR / "x_learner_results.json"

if x_learner_results_path.exists():
    with open(x_learner_results_path) as f:
        x_learner_results = json.load(f)
    x_learner_segment_test = pd.DataFrame(x_learner_results["mens_womens_segment_diagnostic"]).T.astype(float)

    comparison = pd.DataFrame({
        "true_actual_lift_pooled": segment_table_pooled["actual_lift"],
        "x_learner_mean_tau_hat": x_learner_segment_test["mean_tau_hat"],
        "r_learner_mean_tau_hat": segment_table["mean_tau_hat"],
    })
    comparison["x_learner_pct_of_true"] = comparison["x_learner_mean_tau_hat"] / comparison["true_actual_lift_pooled"]
    comparison["r_learner_pct_of_true"] = comparison["r_learner_mean_tau_hat"] / comparison["true_actual_lift_pooled"]
    display(comparison)

    print(f"\nTest Qini -- X-learner: {x_learner_results['qini_score']['X-Learner']:.4f}")

    x_learner_both_consistency = x_learner_results.get("mens_womens_both_train_vs_test_consistency")
    if x_learner_both_consistency is not None:
        print(f"X-learner's own train-vs-test consistency check for 'both': z={x_learner_both_consistency['z']:.2f} "
              f"({'agree' if x_learner_both_consistency['agrees'] else 'disagree'})")
else:
    print("x_learner_results.json not found -- run 06_x_learner.ipynb first for a head-to-head comparison.")
    comparison = None

### Qini curve, cumulative gain curve, AUUC, and Qini score

In [ ]:
tau_hat_dict = {
    "R-Learner": tau_hat_r,
}

scores = compute_qini_auuc(y_test, treatment_test, tau_hat_dict)
print("Qini score:")
print(scores["qini_score"])
print("\nAUUC score:")
print(scores["auuc_score"])

fig_qini = plot_qini_curve(y_test, treatment_test, tau_hat_dict)
save_fig(fig_qini, "r_learner_qini_curve.png")

fig_gain = plot_cumulative_gain_curve(y_test, treatment_test, tau_hat_dict)
save_fig(fig_gain, "r_learner_cumgain_curve.png")

fig_decile_bar = plot_decile_lift_bar(decile_table, title="Actual lift by predicted-uplift decile (R-learner)")
save_fig(fig_decile_bar, "r_learner_decile_lift_bar.png")

### Uplift at top 10%, 20%, and 30%

In [ ]:
top_k_results = pd.DataFrame([
    uplift_at_top_k(y_test, treatment_test, tau_hat_r, k) for k in (0.1, 0.2, 0.3)
]).set_index("k")

display(top_k_results)

### ROI targeting-policy comparison (constant contact volume)

Same fixed-volume methodology as `06_x_learner.ipynb`: hold the number of contacted customers **fixed** at the top 30% (~4,800) and compare two *selection rules* — ranking by predicted response probability vs. ranking by predicted uplift (`tau_hat_r`). Both rules contact the same number of customers at the same email cost, so any difference is due purely to *which* customers each rule selects — isolating the value of uplift modeling from the separate, assumption-dependent question of *how many* customers to email.

**⚠️ Outcome note (documented proxy):** both ranking signals are defined on the project's primary outcome, `visit` — `tau_hat_r` is a *visit*-uplift estimate and the response baseline's `logistic_proba` is a *visit*-probability estimate — while the realized lift/net value below is measured on `conversion` (and `spend`), the business outcome. This is an apples-to-apples *ranking* comparison (both signals visit-based) evaluated on conversion, not a claim that either signal was trained on conversion; it assumes visit-persuadability tracks conversion-persuadability. A conversion-trained uplift model is the proper tool and a documented follow-up.

Dollar assumptions shared identically by both rules (so any `net_value` difference reflects selection, not cost): `email_cost=$0.05` per customer (external assumption, README's illustrative example) and `profit_per_conversion` derived empirically from `spend | conversion==1` on the **train** split (revenue proxy, not true margin — Hillstrom has no cost-of-goods data). The response-probability ranking reuses `03_response_baseline.ipynb`'s committed `logistic_proba` predictions after verifying row alignment with `y_test`.

In [ ]:
response_baseline = pd.read_csv(PREDICTIONS_DIR / "response_baseline_predictions.csv")
assert len(response_baseline) == len(y_test), "response_baseline_predictions.csv row count doesn't match y_test."
assert (response_baseline["y_true"].to_numpy() == y_test.to_numpy()).all(), (
    "response_baseline_predictions.csv rows aren't aligned with y_test -- check it was regenerated "
    "against the same processed test split before trusting response_score_test below."
)
response_score_test = response_baseline["logistic_proba"].to_numpy()

converters_train = secondary_y_train["conversion"] == 1
profit_per_conversion = float(secondary_y_train.loc[converters_train, "spend"].mean())
print(f"Empirical value per conversion (avg spend | conversion==1, train, both arms pooled): ${profit_per_conversion:.2f}")
print(f"  n converters (train): {int(converters_train.sum())}")

roi_table = compare_targeting_strategies(
    conversion=secondary_y_test["conversion"],
    spend=secondary_y_test["spend"],
    tau_hat=tau_hat_r,
    treatment=treatment_test,
    response_score=response_score_test,
    email_cost=0.05,
    profit_per_conversion=profit_per_conversion,
)

display(roi_table)

roi_table.to_csv(TABLES_DIR / "r_learner_roi_comparison.csv")
print("Saved:", TABLES_DIR / "r_learner_roi_comparison.csv")

In [ ]:
# roi_table already IS the constant-volume comparison: both rules target the same
# top 30% at the same email cost, so any difference is purely which customers each
# rule selects. Flag statistical significance (95% CI on the lift excludes zero).
comparison_roi = roi_table.copy()
comparison_roi["significant_at_95pct"] = comparison_roi["incremental_conversion_rate_ci_lower"] > 0

assert comparison_roi["n_targeted"].nunique() == 1, (
    "Constant-volume comparison requires both rules to target the same number of customers."
)

display(
    comparison_roi[[
        "n_targeted",
        "incremental_conversion_rate",
        "incremental_conversion_rate_ci_lower",
        "incremental_conversion_rate_ci_upper",
        "net_value",
        "significant_at_95pct",
    ]]
)

resp = comparison_roi.loc["target_by_response_probability"]
uplift = comparison_roi.loc["target_by_predicted_uplift"]
ratio = uplift["incremental_conversion_rate"] / resp["incremental_conversion_rate"]
print(
    f"At a fixed {uplift['n_targeted']:,.0f}-customer contact volume (top 30%), same email cost:\n"
    f"  response-probability targeting: lift {resp['incremental_conversion_rate']:.3%} "
    f"[{resp['incremental_conversion_rate_ci_lower']:.3%}, {resp['incremental_conversion_rate_ci_upper']:.3%}] "
    f"-- {'significant' if resp['significant_at_95pct'] else 'NOT distinguishable from zero'}\n"
    f"  predicted-uplift targeting:     lift {uplift['incremental_conversion_rate']:.3%} "
    f"[{uplift['incremental_conversion_rate_ci_lower']:.3%}, {uplift['incremental_conversion_rate_ci_upper']:.3%}] "
    f"-- {'significant' if uplift['significant_at_95pct'] else 'NOT distinguishable from zero'}\n"
    f"  => uplift selection captures {ratio:.1f}x the realized conversion lift of response "
    f"selection, at identical contact volume and cost."
)

#### Why this comparison is the case for uplift modeling

The two rows above contact the **same number of customers** (top 30%, ~4,800) at the **same email cost** — the only thing that differs is *which* customers each rule selects, isolating the value of uplift modeling from the separate, assumption-heavy question of *how many* customers to email.

Selecting recipients by **predicted response probability** over-selects "sure things" — customers who would have converted with or without the email — so the incremental effect of contacting them is small. Selecting the same-size group by **predicted uplift** targets persuadable customers where the email actually moves conversion. At a fixed contact budget, that distinction is the difference between a targeting policy that pays off and one that doesn't — the concrete argument for building an uplift model rather than reusing a response-propensity score.

**Scope caveat (see the outcome note above):** both rankers here are *visit*-based while the lift is measured on *conversion*, so the precise claim is that a *visit*-uplift ranking selects a higher realized *conversion*-lift group than a *visit*-response ranking, at equal volume and cost. A conversion-trained uplift model is the documented next step. And as with the X-learner, this R-learner `tau_hat_r` is itself a weaker uplift ranker than the logistic T-learner on this dataset (see `04_t_learner.ipynb` / the cross-model Qini comparison) — the point here is the *response-vs-uplift* selection contrast, not that the R-learner is the best available ranker.

### Save model-level results

Summary metrics, plus run metadata and the head-to-head comparison against X-learner, saved to `outputs/model_results/r_learner_results.json`.

In [ ]:
results = {
    "model": "r_learner",
    "control_name": 0,
    "random_state": RANDOM_STATE,
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "known_propensity": p_known,
    "propensity_check": propensity_check,
    "model_family_selection": {
        "method": f"{N_FOLDS}-fold out-of-fold CV within X_train",
        "winning_family": winning_family,
        "oof_scores": family_comparison.to_dict(orient="index"),
    },
    "mens_womens_segment_diagnostic": segment_table.to_dict(orient="index"),
    "mens_womens_segment_train": segment_table_train.to_dict(orient="index"),
    "mens_womens_segment_pooled": segment_table_pooled.to_dict(orient="index"),
    "mens_womens_both_train_vs_test_consistency": both_consistency,
    "qini_score": scores["qini_score"].to_dict(),
    "auuc_score": scores["auuc_score"].to_dict(),
    "ate_test": float(decile_table.loc["Overall", "actual_lift"]),
    "top_decile_actual_lift": float(decile_table.loc[1, "actual_lift"]),
    "uplift_at_top_k": {
        f"{int(k * 100)}pct": {
            "actual_lift": float(row["actual_lift"]),
            "incremental_outcomes_captured": float(row["incremental_outcomes_captured"]),
        }
        for k, row in top_k_results.iterrows()
    },
    "roi_fixed_volume_comparison": {
        "description": "constant contact volume (top 30%): response-probability vs. predicted-uplift selection, same n and same cost",
        "top_k": 0.3,
        "n_targeted": int(roi_table["n_targeted"].iloc[0]),
        "target_by_response_probability": {
            "incremental_conversion_rate": float(roi_table.loc["target_by_response_probability", "incremental_conversion_rate"]),
            "ci_lower": float(roi_table.loc["target_by_response_probability", "incremental_conversion_rate_ci_lower"]),
            "ci_upper": float(roi_table.loc["target_by_response_probability", "incremental_conversion_rate_ci_upper"]),
            "net_value": float(roi_table.loc["target_by_response_probability", "net_value"]),
        },
        "target_by_predicted_uplift": {
            "incremental_conversion_rate": float(roi_table.loc["target_by_predicted_uplift", "incremental_conversion_rate"]),
            "ci_lower": float(roi_table.loc["target_by_predicted_uplift", "incremental_conversion_rate_ci_lower"]),
            "ci_upper": float(roi_table.loc["target_by_predicted_uplift", "incremental_conversion_rate_ci_upper"]),
            "net_value": float(roi_table.loc["target_by_predicted_uplift", "net_value"]),
        },
    },
    "roi_assumptions": {
        "email_cost": 0.05,
        "email_cost_source": "external assumption (README illustrative example; not observable in the data)",
        "profit_per_conversion": profit_per_conversion,
        "profit_per_conversion_source": "empirical: mean(spend | conversion==1) on train split, both arms pooled -- revenue proxy, not true margin",
    },
}

if comparison is not None:
    results["comparison_vs_x_learner"] = comparison.to_dict(orient="index")

with open(MODEL_RESULTS_DIR / "r_learner_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Saved:", MODEL_RESULTS_DIR / "r_learner_results.json")

### Summary

In [ ]:
control_rate = 1 - p_known
ratio = p_known / control_rate

resp_lift = roi_table.loc["target_by_response_probability", "incremental_conversion_rate"]
uplift_lift = roi_table.loc["target_by_predicted_uplift", "incremental_conversion_rate"]

print(f"Observed treated share: {p_known:.1%}  |  control share: {control_rate:.1%}  |  treated is {ratio:.2f}x control")
print(f"Winning model family: {winning_family}")
print(f"Qini score: {scores['qini_score']['R-Learner']:.4f}")
print(f"AUUC score: {scores['auuc_score']['R-Learner']:.4f}")
print(f"Top-decile actual lift: {decile_table.loc[1, 'actual_lift']:.4f}  |  overall ATE: {decile_table.loc['Overall', 'actual_lift']:.4f}")
print(f"Both-segment predicted uplift: {segment_table.loc['both', 'mean_tau_hat']:.4f}  |  true (pooled train+test): {segment_table_pooled.loc['both', 'actual_lift']:.4f}")
print(f"Fixed-volume (top 30%) selection -- uplift lift {uplift_lift:.3%} vs response lift {resp_lift:.3%} "
      f"({uplift_lift / resp_lift:.1f}x); better rule: {roi_table['net_value'].idxmax()}")